# RQ3: Meta-Learning Best Generator Prediction

**Research Question 3 (RQ3):** *Can a meta-model predict the best generator from dataset meta-features alone?*

This notebook analyzes the leave-one-dataset-out (LODO) evaluation of the RandomForest meta-predictor
trained on OpenML-CC18 dataset profiles (`general`, `statistical`, and `info-theory` meta-features via `pymfe`)
to predict the generator that achieves the highest balanced fidelity-utility-privacy score among candidates
that pass the fail-closed privacy gate.

> **Note:** RQ3 is framed as exploratory. With small-to-medium benchmark suites (~25 datasets),
> honest reporting of both positive and negative results vs the majority baseline is essential.

In [ ]:
import os
import json
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import joblib

# Path configuration
STORE_ROOT = Path(os.environ.get("STL_STORAGE_ROOT", "../outputs/store"))
LODO_PATH = STORE_ROOT / "meta" / "lodo.json"
PREDICTOR_JSON = STORE_ROOT / "meta" / "predictor.json"
PREDICTOR_JOBLIB = STORE_ROOT / "meta" / "predictor.joblib"

print(f"Store root: {STORE_ROOT.resolve()}")
print(f"LODO results exist: {LODO_PATH.exists()}")
print(f"Predictor model exists: {PREDICTOR_JOBLIB.exists()}")

## 1. Load LODO Evaluation and Summary

We read the LODO results generated by `experiments/run.py meta-eval`.

In [ ]:
if not LODO_PATH.exists():
    raise FileNotFoundError(f"{LODO_PATH} not found. Run `experiments/run.py meta-eval` first.")

with open(LODO_PATH, "r", encoding="utf-8") as f:
    lodo = json.load(f)

with open(PREDICTOR_JSON, "r", encoding="utf-8") as f:
    meta_summary = json.load(f)

metrics_df = pd.DataFrame([{
    "Metric": "Accuracy",
    "Meta-Model": lodo["accuracy"],
    "Majority Baseline": lodo["baseline_accuracy"],
}, {
    "Metric": "Balanced Accuracy",
    "Meta-Model": lodo["balanced_accuracy"],
    "Majority Baseline": lodo["baseline_balanced_accuracy"],
}, {
    "Metric": "Mean Regret (lower is better)",
    "Meta-Model": lodo["mean_regret"],
    "Majority Baseline": lodo["baseline_mean_regret"],
}])

print(f"Total usable datasets in LODO: {len(lodo['predictions'])}")
print(f"Datasets where all generators failed the gate: {lodo['n_none']}")
print(f"Total meta-features used: {len(lodo['features'])}")
metrics_df

## 2. Per-Dataset LODO Predictions

Let's inspect the held-out predictions and individual regret values for each dataset.

In [ ]:
preds_df = pd.DataFrame(lodo["predictions"])
preds_df

## 3. Accuracy and Regret Comparison

We compare the meta-predictor against the baseline (which always predicts the most common winner in training folds).

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))

# Accuracy comparison
acc_vals = [lodo["accuracy"] or 0.0, lodo["baseline_accuracy"] or 0.0]
axes[0].bar(["Meta-Model", "Baseline"], acc_vals, color=["#2563eb", "#94a3b8"], width=0.5)
axes[0].set_ylabel("Accuracy")
axes[0].set_ylim(0, 1.0)
axes[0].set_title("LODO Accuracy vs Majority Baseline")
for i, v in enumerate(acc_vals):
    axes[0].text(i, v + 0.02, f"{v:.2f}", ha="center", fontweight="bold")

# Mean Regret comparison (lower is better)
reg_vals = [lodo["mean_regret"] or 0.0, lodo["baseline_mean_regret"] or 0.0]
axes[1].bar(["Meta-Model", "Baseline"], reg_vals, color=["#10b981", "#f59e0b"], width=0.5)
axes[1].set_ylabel("Mean Regret (lower is better)")
axes[1].set_title("Mean Regret vs Majority Baseline")
for i, v in enumerate(reg_vals):
    axes[1].text(i, v + 0.005, f"{v:.3f}", ha="center", fontweight="bold")

plt.tight_layout()
plt.show()

## 4. Meta-Feature Importances

Which dataset characteristics (e.g., sample size, correlation, mutual information, class entropy) most influence the recommended generator?

In [ ]:
if PREDICTOR_JOBLIB.exists():
    artifact = joblib.load(PREDICTOR_JOBLIB)
    rf = artifact["model"]
    features = artifact["features"]
    importances = rf.feature_importances_
    
    imp_df = pd.DataFrame({"Feature": features, "Importance": importances})
    top_imp = imp_df.sort_values(by="Importance", ascending=False).head(15)
    
    plt.figure(figsize=(9, 5))
    plt.barh(top_imp["Feature"][::-1], top_imp["Importance"][::-1], color="#3b82f6")
    plt.xlabel("Gini Importance")
    plt.title("Top 15 Most Informative Dataset Meta-Features")
    plt.tight_layout()
    plt.show()
else:
    print("Predictor artifact not found.")

## 5. Confusion Matrix

Cross-tabulation of true best generator vs predicted best generator across the benchmark folds.

In [ ]:
if lodo["predictions"]:
    y_true = [p["true"] for p in lodo["predictions"]]
    y_pred = [p["predicted"] for p in lodo["predictions"]]
    labels = sorted(list(set(y_true) | set(y_pred)))
    
    cm = confusion_matrix(y_true, y_pred, labels=labels)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=labels)
    
    fig, ax = plt.subplots(figsize=(6, 5))
    disp.plot(ax=ax, cmap="Blues", colorbar=False)
    plt.title("LODO Confusion Matrix (Predicted vs True Winner)")
    plt.tight_layout()
    plt.show()
else:
    print("No predictions available to plot confusion matrix.")

## 6. Discussion and Future Work

### Summary of Findings
- **Regret Reduction:** Mean regret measures how much balanced score is lost if a practitioner follows the meta-model suggestion compared to the true ex-post optimal generator. A meta-model that achieves lower regret than the majority baseline provides practical utility even when raw classification accuracy is modest.
- **Gate Failures & Low-Resource Regimes:** In high-privacy settings, complex deep generative models frequently fail privacy safeguards (linkability, inference, or membership inference advantage). Simpler statistical copulas or DP-synthesizers (e.g. MST) often emerge as robust winners on smaller tabular profiles.
- **Exploratory Framing:** Meta-learning performance scales with the diversity and number of benchmarked datasets. Running the full 25-dataset sweep on owner compute (`scripts/run_meta_benchmark.sh`) expands the training folds and stabilizes meta-feature distributions.